In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

con = duckdb.connect()


In [27]:
# RUTA = "data/transactions/year=2023/**/*.csv"
table_name = "transactions"
RUTA = f"data/{table_name}/year=2025/month=10/**/*.csv"
# RUTA = f"data/{table_name}.csv"

# Vista sobre todos los archivos: agrega year, month, day y el archivo de origen
con.sql(f"""
    CREATE OR REPLACE VIEW '{table_name}' AS
    SELECT *
    FROM read_csv('{RUTA}',
                  hive_partitioning = true,
                  union_by_name = true,
                  filename = true)
""")


In [25]:
# -- 1) Qué pares de monedas trae la tabla de tasas y en qué dirección
con.sql("""SELECT source_currency, target_currency, count(*) AS dias,
       min(date) AS desde, max(date) AS hasta,
       round(avg(exchange_rate), 6) AS tasa_promedio
FROM daily_exchange_rates
GROUP BY ALL ORDER BY 1, 2;""").show()



┌─────────────────┬─────────────────┬───────┬────────────┬────────────┬───────────────┐
│ source_currency │ target_currency │ dias  │   desde    │   hasta    │ tasa_promedio │
│     varchar     │     varchar     │ int64 │    date    │    date    │    double     │
├─────────────────┼─────────────────┼───────┼────────────┼────────────┼───────────────┤
│ ARS             │ COP             │  1097 │ 2023-06-17 │ 2026-06-17 │     11.435088 │
│ ARS             │ MXN             │  1097 │ 2023-06-17 │ 2026-06-17 │       0.04857 │
│ ARS             │ USD             │  1097 │ 2023-06-17 │ 2026-06-17 │      0.002859 │
│ COP             │ ARS             │  1097 │ 2023-06-17 │ 2026-06-17 │      0.087433 │
│ COP             │ MXN             │  1097 │ 2023-06-17 │ 2026-06-17 │      0.004247 │
│ COP             │ USD             │  1097 │ 2023-06-17 │ 2026-06-17 │       0.00025 │
│ MXN             │ ARS             │  1097 │ 2023-06-17 │ 2026-06-17 │     20.591067 │
│ MXN             │ COP         

In [28]:


# -- 2) ¿amount_usd coincide con la tasa del día de la transacción?
con.sql("""
WITH t AS (
  SELECT CAST(transaction_date AS DATE) AS d, currency, amount, amount_usd
  FROM transactions TABLESAMPLE 5%
  WHERE currency <> 'USD'
)
SELECT t.currency,
       count(*)                                   AS n,
       count(*) FILTER (WHERE amount_usd IS NULL) AS sin_amount_usd,
       count(r.exchange_rate)                     AS con_tasa_ese_dia,
       round(median(abs(amount_usd - amount * r.exchange_rate) / amount_usd), 4) AS error_si_multiplica,
       round(median(abs(amount_usd - amount / r.exchange_rate) / amount_usd), 4) AS error_si_divide
FROM t
LEFT JOIN daily_exchange_rates r
  ON r.date = t.d
 AND ((r.source_currency = t.currency AND r.target_currency = 'USD')
   OR (r.source_currency = 'USD' AND r.target_currency = t.currency))
GROUP BY 1;""")

┌──────────┬───────┬────────────────┬──────────────────┬─────────────────────┬─────────────────┐
│ currency │   n   │ sin_amount_usd │ con_tasa_ese_dia │ error_si_multiplica │ error_si_divide │
│ varchar  │ int64 │     int64      │      int64       │       double        │     double      │
├──────────┼───────┼────────────────┼──────────────────┼─────────────────────┼─────────────────┤
│ ARS      │  1316 │             44 │             1316 │           60552.466 │      60929.3699 │
│ COP      │  2024 │            120 │             2024 │        7927770.7972 │    7871539.2222 │
└──────────┴───────┴────────────────┴──────────────────┴─────────────────────┴─────────────────┘

In [21]:
con.sql(f"""
    SELECT *
    FROM {table_name} LIMIT 10
""").show()

┌──────────────────────────┬──────────────────────┬──────────────┬──────────────────┬────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────┬─────────────────┬───────────────────┬──────────────────

In [13]:
con.sql(f"""SELECT distinct(product_type)
               FROM {table_name} LIMIT 100""")


┌──────────────────────┐
│     product_type     │
│       varchar        │
├──────────────────────┤
│ Tarjeta Crédito      │
│ Inversión            │
│ Préstamo Hipotecario │
│ Seguro               │
│ Cuenta Corriente     │
│ Tarjeta Débito       │
│ Cuenta Ahorro        │
│ Préstamo Personal    │
└──────────────────────┘

In [14]:
con.sql(f"""SELECT *
               FROM {table_name} LIMIT 100""")

┌──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────┬─────────────────┬──────────────┬───────────────┬──────────────┬─────────────────┬───────────────────┬────────────────┬─────────────────┬────────────────┬───────────────┬───────────────────────┬─────────────────────┬───────────────────┐
│    product_id    │   customer_id    │   product_type   │  product_number  │ currency │ current_balance │ credit_limit │ interest_rate │ opening_date │ expiration_date │ opening_branch_id │ product_status │ opening_channel │ has_linked_app │ days_past_due │ last_transaction_date │    last_updated     │     filename      │
│     varchar      │     varchar      │     varchar      │     varchar      │ varchar  │     double      │    double    │    double     │     date     │      date       │      varchar      │    varchar     │     varchar     │    boolean     │    double     │       timestamp       │      timestamp      │      varchar      │
├──────────────────┼─────

In [3]:

print("=== MUESTRA ===")
con.sql(f"""SELECT category, subcategory, reception_channel,
  description, claimed_amount, priority, status,resolution, compensation_granted,
  resolution_satisfaction, is_repeat_complainer AS resume
               FROM {table_name} LIMIT 100""")




=== MUESTRA ===


┌──────────────┬──────────────────────┬───────────────────┬────────────────────────────────────┬────────────────┬──────────┬────────────┬────────────────────────────────────────────────────────────────────────────────────┬──────────────────────┬─────────────────────────┬─────────┐
│   category   │     subcategory      │ reception_channel │            description             │ claimed_amount │ priority │   status   │                                     resolution                                     │ compensation_granted │ resolution_satisfaction │ resume  │
│   varchar    │       varchar        │      varchar      │              varchar               │     double     │ varchar  │  varchar   │                                      varchar                                       │        double        │         double          │ boolean │
├──────────────┼──────────────────────┼───────────────────┼────────────────────────────────────┼────────────────┼──────────┼────────────┼─────────────────

In [5]:
categoricas = ["case_type", "category", "subcategory", "reception_channel", "description",
               "priority", "status", "resolution_satisfaction", "is_repeat_complainer",
               "sla_breached", "currency"]

# # Cuántos valores distintos y cuántos nulos tiene cada columna
# for col in categoricas:
#     con.sql(f"""
#         SELECT '{col}' AS columna,
#                COUNT(DISTINCT {col}) AS valores_unicos,
#                COUNT(*) - COUNT({col}) AS nulos
#         FROM {table_name}
#     """).show()

# Frecuencia de cada valor
for col in categoricas:
    print(f"\n=== {col} ===")
    con.sql(f"""
        SELECT {col}, COUNT(*) AS n,
               ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
        FROM {table_name}
        GROUP BY {col} ORDER BY n DESC
    """).show(max_rows=50)


=== case_type ===
┌────────────┬───────┬────────┐
│ case_type  │   n   │  pct   │
│  varchar   │ int64 │ double │
├────────────┼───────┼────────┤
│ Complaint  │ 40452 │   60.3 │
│ Claim      │ 16598 │   24.7 │
│ Request    │  6761 │   10.1 │
│ Suggestion │  3284 │    4.9 │
└────────────┴───────┴────────┘


=== category ===
┌──────────────┬───────┬────────┐
│   category   │   n   │  pct   │
│   varchar    │ int64 │ double │
├──────────────┼───────┼────────┤
│ Transactions │ 13580 │   20.2 │
│ Fees         │ 13553 │   20.2 │
│ Technical    │ 13407 │   20.0 │
│ Branch       │ 13361 │   19.9 │
│ Service      │ 13194 │   19.7 │
└──────────────┴───────┴────────┘


=== subcategory ===
┌──────────────────────┬───────┬────────┐
│     subcategory      │   n   │  pct   │
│       varchar        │ int64 │ double │
├──────────────────────┼───────┼────────┤
│ Cargo no reconocido  │ 12297 │   18.3 │
│ Cobro indebido       │ 12194 │   18.2 │
│ Problema con app     │ 12128 │   18.1 │
│ Atención en sucu

In [6]:
con.sql(f"""SELECT COUNT(*) AS filas, COUNT(DISTINCT complaint_id) AS ids_unicos
            FROM {table_name}""").show()

┌───────┬────────────┐
│ filas │ ids_unicos │
│ int64 │   int64    │
├───────┼────────────┤
│ 67095 │      67095 │
└───────┴────────────┘



In [32]:
#Qué categorías son disputas de transacciones? Define el alcance del agente y las etiquetas de B.
con.sql(f"""
    SELECT category, subcategory, COUNT(*) AS n
    FROM {table_name}
    GROUP BY ALL ORDER BY n DESC
""").show(max_rows=100)

┌──────────────┬──────────────────────┬───────┐
│   category   │     subcategory      │   n   │
│   varchar    │       varchar        │ int64 │
├──────────────┼──────────────────────┼───────┤
│ Transactions │ Cargo no reconocido  │ 12297 │
│ Fees         │ Cobro indebido       │ 12194 │
│ Technical    │ Problema con app     │ 12128 │
│ Branch       │ Atención en sucursal │ 11892 │
│ Service      │ Calidad de servicio  │ 11886 │
│ Branch       │ NULL                 │  1469 │
│ Fees         │ NULL                 │  1359 │
│ Service      │ NULL                 │  1308 │
│ Transactions │ NULL                 │  1283 │
│ Technical    │ NULL                 │  1279 │
└──────────────┴──────────────────────┴───────┘
  10 rows                           3 columns



In [ ]:
# que casos terminan escalados? Qué porcentaje de cada prioridad termina escalado?
con.sql(f"""
    SELECT priority, COUNT(*) AS n,
           ROUND(100.0 * AVG(CASE WHEN status = 'Escalated' THEN 1 ELSE 0 END), 1) AS pct_escalado
    FROM {table_name}
    GROUP BY priority ORDER BY pct_escalado DESC
""").show()
# La prioridad no determina la escalada, aunque los casos de prioridad alta tienen un porcentaje más alto de escalada

┌──────────┬───────┬──────────────┐
│ priority │   n   │ pct_escalado │
│ varchar  │ int64 │    double    │
├──────────┼───────┼──────────────┤
│ Critical │  3355 │          5.5 │
│ Medium   │ 33439 │          5.0 │
│ Low      │ 20411 │          4.9 │
│ High     │  9890 │          4.6 │
└──────────┴───────┴──────────────┘



In [9]:
con.sql(f"""
    SELECT distinct (resolution), COUNT(*) AS n
    FROM {table_name}
    WHERE resolution IS NOT NULL
""").show()

┌────────────────────────────────────────────────────────────────────────────────────┐
│                                     resolution                                     │
│                                      varchar                                       │
├────────────────────────────────────────────────────────────────────────────────────┤
│ Se brindó explicación detallada al cliente y se resolvió la situación.             │
│ Se escaló a área correspondiente y se aplicó la solución definitiva.               │
│ Se verificó la información y se procedió con la corrección solicitada.             │
│ Se otorgó compensación al cliente por las molestias ocasionadas.                   │
│ Se revisó el caso y se realizó el ajuste correspondiente en la cuenta del cliente. │
└────────────────────────────────────────────────────────────────────────────────────┘



In [ ]:
# los clientes reicidentes se escalan mas?
con.sql(f"""
    SELECT is_repeat_complainer, COUNT(*) AS n,
           ROUND(100.0 * AVG(CASE WHEN status = 'Escalated' THEN 1 ELSE 0 END), 1) AS pct_escalado
    FROM {table_name} GROUP BY 1
""").show()
# La escalada no depende de si el cliente es reincidente o no

┌──────────────────────┬───────┬──────────────┐
│ is_repeat_complainer │   n   │ pct_escalado │
│       boolean        │ int64 │    double    │
├──────────────────────┼───────┼──────────────┤
│ false                │ 57009 │          5.0 │
│ true                 │ 10086 │          4.9 │
└──────────────────────┴───────┴──────────────┘



In [ ]:
# El canal importa? Qué porcentaje de cada canal termina escalado?
con.sql(f"""
    SELECT reception_channel, COUNT(*) AS n,
           ROUND(100.0 * AVG(CASE WHEN status = 'Escalated' THEN 1 ELSE 0 END), 1) AS pct_escalado,
           ROUND(100.0 * AVG(CASE WHEN TRY_CAST(sla_breached AS BOOLEAN) THEN 1 ELSE 0 END), 1) AS pct_sla_incumplido
    FROM {table_name} GROUP BY 1 ORDER BY pct_escalado DESC
""").show()
# La escalada no depende del canal

┌───────────────────┬───────┬──────────────┬────────────────────┐
│ reception_channel │   n   │ pct_escalado │ pct_sla_incumplido │
│      varchar      │ int64 │    double    │       double       │
├───────────────────┼───────┼──────────────┼────────────────────┤
│ Branch            │  2683 │          5.4 │               19.9 │
│ Regulator         │   717 │          5.4 │               22.5 │
│ App               │  6727 │          5.2 │               20.5 │
│ Email             │ 13323 │          4.9 │               19.8 │
│ Web               │  9884 │          4.9 │               20.4 │
│ Call Center       │ 33761 │          4.9 │               20.1 │
└───────────────────┴───────┴──────────────┴────────────────────┘



In [36]:
#¿Cuánto tarda y cuánto se compensa por categoría
con.sql(f"""
    SELECT category, COUNT(*) AS n,
           ROUND(MEDIAN(TRY_CAST(resolution_days AS INT)), 0) AS dias_mediana,
           ROUND(100.0 * AVG(CASE WHEN TRY_CAST(compensation_granted AS DOUBLE) > 0 THEN 1 ELSE 0 END), 1) AS pct_compensado,
           ROUND(AVG(TRY_CAST(resolution_satisfaction AS INT)), 2) AS satisfaccion
    FROM {table_name} GROUP BY 1 ORDER BY n DESC
""").show()

┌──────────────┬───────┬──────────────┬────────────────┬──────────────┐
│   category   │   n   │ dias_mediana │ pct_compensado │ satisfaccion │
│   varchar    │ int64 │    double    │     double     │    double    │
├──────────────┼───────┼──────────────┼────────────────┼──────────────┤
│ Transactions │ 13580 │         15.0 │            7.3 │         3.09 │
│ Fees         │ 13553 │         16.0 │            7.0 │         2.97 │
│ Technical    │ 13407 │         16.0 │            6.7 │         3.02 │
│ Branch       │ 13361 │         16.0 │            7.0 │         3.05 │
│ Service      │ 13194 │         16.0 │            6.6 │         2.97 │
└──────────────┴───────┴──────────────┴────────────────┴──────────────┘



In [ ]:
con.sql(f"""
    SELECT distinct(description) AS filas,
            COUNT(DISTINCT description) AS textos_distintos,
            ROUND(100.0 * COUNT(DISTINCT description) / COUNT(*), 1) AS pct_distintos
    FROM {table_name}
""").show()

┌───────┬──────────────────┬───────────────┐
│ filas │ textos_distintos │ pct_distintos │
│ int64 │      int64       │    double     │
├───────┼──────────────────┼───────────────┤
│ 67095 │                5 │           0.0 │
└───────┴──────────────────┴───────────────┘

